##  自然言語処理による感情分析（BERT）

- 本ファイルでは、フォルダの作成とファイルのダウンロードを行います。

In [1]:
import os
import urllib.request
import zipfile
import tarfile
import glob
import io

In [2]:
print(os.getcwd()) # 現在のディレクトリ

/Users/miku.yanagimoto/Projects/nlp-sentiment-bert


In [3]:
# フォルダ「data」が存在しない場合は作成する
data_dir = "./data/"
if not os.path.exists(data_dir):
    os.mkdir(data_dir)
print(os.path.join(os.getcwd(), data_dir))

/Users/miku.yanagimoto/Projects/nlp-sentiment-bert/./data/


In [4]:
# フォルダ「vocab」が存在しない場合は作成する
vocab_dir = "./vocab/"
if not os.path.exists(vocab_dir):
    os.mkdir(vocab_dir)
print(os.path.join(os.getcwd(), vocab_dir))

/Users/miku.yanagimoto/Projects/nlp-sentiment-bert/./vocab/


In [5]:
# フォルダ「weights」が存在しない場合は作成する
weights_dir = "./weights/"
if not os.path.exists(weights_dir):
    os.mkdir(weights_dir)
print(os.path.join(os.getcwd(), weights_dir))

/Users/miku.yanagimoto/Projects/nlp-sentiment-bert/./weights/


In [6]:
# 単語集：ボキャブラリーをダウンロード

# 'bert-base-uncased': 
# https://s3.amazonaws.com/models.huggingface.co/bert/bert-base-uncased-vocab.txt

save_path="./vocab/bert-base-uncased-vocab.txt"
url = "https://s3.amazonaws.com/models.huggingface.co/bert/bert-base-uncased-vocab.txt"
urllib.request.urlretrieve(url, save_path)

('./vocab/bert-base-uncased-vocab.txt',
 <http.client.HTTPMessage at 0x1067c7d60>)

In [7]:
# BERTの学習済みモデル 'bert-base-uncased'
# https://github.com/huggingface/pytorch-pretrained-BERT/
# https://s3.amazonaws.com/models.huggingface.co/bert/bert-base-uncased.tar.gz

# ダウンロード
save_path = "./weights/bert-base-uncased.tar.gz"
url = "https://s3.amazonaws.com/models.huggingface.co/bert/bert-base-uncased.tar.gz"
urllib.request.urlretrieve(url, save_path)

# 解凍
archive_file = "./weights/bert-base-uncased.tar.gz"  # Uncasedは小文字化モードという意味です
tar = tarfile.open(archive_file, 'r:gz')
tar.extractall('./weights/')  # 解凍
tar.close()  # ファイルをクローズ

# フォルダ「weights」に「pytorch_model.bin」と「bert_config.json」ができます

# IMDbデータセットをダウンロードし、tsvファイルに整形

元の配布元 http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz は接続が拒否されるため、同じアーカイブのミラーを使います。

In [3]:
# IMDbデータセットをダウンロード。30秒ほどでダウンロードできます
target_dir_path="./data/"

if not os.path.exists(target_dir_path):
    os.mkdir(target_dir_path)
    
# ai.stanford.edu は接続が拒否されるため、同じ aclImdb_v1.tar.gz のミラーを使う
url = "https://github.com/spenceforce/NLP-Simple-to-Spectacular/releases/download/aclImdb_dataset/aclImdb_v1.tar.gz"
save_path = "./data/aclImdb_v1.tar.gz"
urllib.request.urlretrieve(url, save_path)

# './data/aclImdb_v1.tar.gz'の解凍　1分ほどかかります

# tarファイルを読み込み
tar = tarfile.open('./data/aclImdb_v1.tar.gz')
tar.extractall('./data/')  # 解凍
tar.close()  # ファイルをクローズ

# フォルダ「data」内にフォルダ「aclImdb」というものができます。

In [27]:
# IMDbの個別ファイルをtsvにまとめる
target_dir_path="./data/aclImdb/"

if os.path.exists(target_dir_path):
    
    # 訓練データの作成
    print("Create train data")
    f=open('./data/IMDb_train.tsv','w')
    
    # positive dataの作成
    path = './data/aclImdb/train/pos/'
    print(f"path == {path}")
    i = 0
    for fname in glob.glob(os.path.join(path,'*.txt')):
        with io.open(fname, 'r', encoding="utf-8") as ff:
            text = ff.readline()
            if i == 0:
                print(f"fname == {fname}")
                print(f"type(text) == {type(text)}")
                print(f"text[:100] == {text[:100]}")
            # タブがあれば消しておきます
            text = text.replace('\t', " ")
            text = text+'\t'+'1'+'\t'+'\n'
            f.write(text)
        i += 1

    print("--------------------------------")
    # negative dataの作成
    path = './data/aclImdb/train/neg/'
    print(f"path == {path}")
    i = 0
    for fname in glob.glob(os.path.join(path,'*.txt')):
        with io.open(fname, 'r', encoding="utf-8") as ff:
            text = ff.readline()
            if i == 0:
                print(f"fname == {fname}")
                print(f"type(text) == {type(text)}")
                print(f"text[:100] == {text[:100]}")
            # タブがあれば消しておきます
            text = text.replace('\t', " ")
            text = text+'\t'+'0'+'\t'+'\n'
            f.write(text)
            
        i += 1

    f.close()
    
    print("===============================")
    print("Create test data")
    # テストデータの作成
    f=open('./data/IMDb_test.tsv','w')

    # positive dataの作成
    path = './data/aclImdb/test/pos/'
    print(f"path == {path}")

    i = 0
    for fname in glob.glob(os.path.join(path,'*.txt')):
        with io.open(fname, 'r', encoding="utf-8") as ff:
            text = ff.readline()
            
            # タブがあれば消しておきます
            text = text.replace('\t', " ")
        
            text = text+'\t'+'1'+'\t'+'\n'
            f.write(text)
        
        i += 1
    print("--------------------------------")
    # negative dataの作成
    path = './data/aclImdb/test/neg/'
    print(f"path == {path}")
      
    i = 0
    for fname in glob.glob(os.path.join(path,'*.txt')):
        with io.open(fname, 'r', encoding="utf-8") as ff:
            text = ff.readline()
            
            # タブがあれば消しておきます
            text = text.replace('\t', " ")
            
            text = text+'\t'+'0'+'\t'+'\n'
            f.write(text)
    
        i += 1

    f.close()

Create train data
path == ./data/aclImdb/train/pos/
fname == ./data/aclImdb/train/pos/4715_9.txt
type(text) == <class 'str'>
text[:100] == For a movie that gets no respect there sure are a lot of memorable quotes listed for this gem. Imagi
--------------------------------
path == ./data/aclImdb/train/neg/
fname == ./data/aclImdb/train/neg/1821_4.txt
type(text) == <class 'str'>
text[:100] == Working with one of the best Shakespeare sources, this film manages to be creditable to it's source,
Create test data
path == ./data/aclImdb/test/pos/
--------------------------------
path == ./data/aclImdb/test/neg/


以上